# Step 0 — Estimate calibration data audit

Answers one question: **do we have enough data to build per-category and per-person×category calibration multipliers** for the Estimate Calibration feature mocked up in `mockups/estimate-calibration.html`?

This notebook is **read-only**. Nothing is written to AGOL.

What it reports:
1. Overall feasibility — how many completed tasks have both `hours` (est) and `hours_worked` (actual) populated, last 12 months.
2. By category — multiplier (actual÷est) per category, with confidence tier based on sample size.
3. By assignee — same shape.
4. Per-person × category cell coverage — how many heatmap cells have n≥5.
5. Outliers — tasks with multipliers >10× or <0.1× (likely data entry errors).
6. Project-level fallback — if task-level data is thin, can we calibrate at project level instead?

**Decision rule:** if a category has n≥10 with both fields, we treat it as High confidence; 5–9 = Medium; <5 = Insufficient. We need at least 5–6 categories at High confidence and a usable team-overall multiplier for Phase 1 to be worth shipping.

In [ ]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayer
import pandas as pd
import datetime as dt

gis = GIS("home")
print(f"Authenticated as: {gis.users.me.username} @ {gis.url}")

## Config

Time window and confidence thresholds match what the Phase 1 UI mockup uses. Adjust `WINDOW_MONTHS` if you want to see how cutting it down to 6mo or extending to 24mo changes the picture.

In [ ]:
SOURCE_TASKS_URL    = "https://services3.arcgis.com/9coHY2fvuFjG9HQX/ArcGIS/rest/services/datateam_portfolio_v2/FeatureServer/1"
SOURCE_PROJECTS_URL = "https://services3.arcgis.com/9coHY2fvuFjG9HQX/ArcGIS/rest/services/datateam_portfolio_v2/FeatureServer/0"

WINDOW_MONTHS = 12
N_HIGH = 10
N_MED  = 5
OUTLIER_HI = 10.0
OUTLIER_LO = 0.1

CUTOFF = pd.Timestamp(dt.datetime.utcnow()) - pd.DateOffset(months=WINDOW_MONTHS)
print(f"Window: last {WINDOW_MONTHS} months  (cutoff = {CUTOFF.date()})")
print(f"Confidence: n>={N_HIGH} High, {N_MED}-{N_HIGH-1} Medium, <{N_MED} Insufficient")
print(f"Outlier flags: multiplier > {OUTLIER_HI}x or < {OUTLIER_LO}x")

## 1 — Pull tasks (defensive)

Prints the layer's actual field list first (in case schema has drifted from what we expected), then queries with `where="1=1"` + `out_fields="*"` so any SQL or field-name issue surfaces as a clear message rather than a generic 400. Filters to completed + within window in pandas.

In [ ]:
tasks_layer = FeatureLayer(SOURCE_TASKS_URL, gis=gis)

print("Tasks layer fields:")
for f in tasks_layer.properties.fields:
    print(f"  {f['name']:28s} {f['type']}")
print()

fset = tasks_layer.query(where="1=1", out_fields="*", return_geometry=False)
tasks = pd.DataFrame([f.attributes for f in fset.features])
print(f"Pulled {len(tasks)} total tasks.")
print(f"Columns: {list(tasks.columns)}\n")

required = ["actual_end", "hours", "hours_worked", "category", "assignee", "task_number"]
missing = [c for c in required if c not in tasks.columns]
if missing:
    print(f"WARNING: missing expected fields: {missing}")
    print("Substring-match candidates from this layer:")
    for m in missing:
        norm = m.replace("_", "").lower()
        hits = [c for c in tasks.columns if norm in c.replace("_", "").lower()]
        print(f"  {m:18s} -> {hits}")
    raise KeyError(f"Update field name(s) in this and subsequent cells: {missing}")

tasks["actual_end_dt"] = pd.to_datetime(tasks["actual_end"], unit="ms", errors="coerce")
tasks["hours"]         = pd.to_numeric(tasks["hours"],        errors="coerce").fillna(0)
tasks["hours_worked"]  = pd.to_numeric(tasks["hours_worked"], errors="coerce").fillna(0)
tasks["category"]      = tasks["category"].fillna("(uncategorized)")
tasks["assignee"]      = tasks["assignee"].fillna("(unassigned)")

n_with_end = tasks["actual_end_dt"].notna().sum()
recent = tasks[tasks["actual_end_dt"].notna() & (tasks["actual_end_dt"] >= CUTOFF)].reset_index(drop=True)
print(f"Tasks with actual_end set: {n_with_end}")
print(f"After {WINDOW_MONTHS}mo window filter: {len(recent)} tasks")

## 2 — Overall feasibility

The headline number is **% of completed tasks with BOTH `hours` and `hours_worked` > 0**. If this is <20% the whole route is shaky and we should fall back to project-level calibration (Section 6).

In [ ]:
n_total       = len(recent)
n_est_only    = ((recent["hours"]        > 0) & (recent["hours_worked"] == 0)).sum()
n_actual_only = ((recent["hours"]        == 0) & (recent["hours_worked"] > 0)).sum()
n_both        = ((recent["hours"]        > 0) & (recent["hours_worked"] > 0)).sum()
n_neither     = ((recent["hours"]        == 0) & (recent["hours_worked"] == 0)).sum()

print(f"Completed tasks last {WINDOW_MONTHS}mo: {n_total}")
print(f"  Both fields populated:   {n_both:5d}  ({100*n_both/max(n_total,1):5.1f}%)  <- usable for calibration")
print(f"  Estimate only (no log):  {n_est_only:5d}  ({100*n_est_only/max(n_total,1):5.1f}%)")
print(f"  Actual only  (no est):   {n_actual_only:5d}  ({100*n_actual_only/max(n_total,1):5.1f}%)")
print(f"  Neither populated:       {n_neither:5d}  ({100*n_neither/max(n_total,1):5.1f}%)")

usable = recent[(recent["hours"] > 0) & (recent["hours_worked"] > 0)].copy()
usable["mult"] = usable["hours_worked"] / usable["hours"]

if n_both > 0:
    team_mult_weighted = usable["hours_worked"].sum() / usable["hours"].sum()
    team_mult_median   = usable["mult"].median()
    print(f"\nTeam multiplier (hours-weighted): {team_mult_weighted:.2f}x")
    print(f"Team multiplier (median task):    {team_mult_median:.2f}x")
    print(f"Total estimated hours: {usable['hours'].sum():,.0f}h")
    print(f"Total actual hours:    {usable['hours_worked'].sum():,.0f}h")

print("\nVerdict:")
if n_both >= 100:
    print(f"  [OK] Strong signal ({n_both} usable tasks). Phase 1 is feasible as designed.")
elif n_both >= 50:
    print(f"  [~] Moderate signal ({n_both} usable tasks). Expect more 'Insufficient' cells; team-overall multiplier still credible.")
elif n_both >= 20:
    print(f"  [!] Thin signal ({n_both} usable tasks). Per-category may be too sparse; recommend project-level fallback.")
else:
    print(f"  [NO] Insufficient ({n_both} usable tasks). Step 1 should be a data-entry push, not a UI build.")

## 3 — By category

What the main Phase 1 table on the Insights tab would show. The `confidence` column maps directly to the green/yellow/gray chip in the mockup.

In [ ]:
def confidence_tier(n):
    if n >= N_HIGH: return "High"
    if n >= N_MED:  return "Medium"
    return "Insufficient"

grp = usable.groupby("category").agg(
    n=("task_number", "count"),
    sum_est=("hours", "sum"),
    sum_actual=("hours_worked", "sum"),
    median_mult=("mult", "median"),
).reset_index()
grp["weighted_mult"] = grp["sum_actual"] / grp["sum_est"].replace(0, pd.NA)
grp["confidence"] = grp["n"].apply(confidence_tier)
grp = grp.sort_values("n", ascending=False).reset_index(drop=True)

pd.set_option("display.max_rows", 50)
pd.set_option("display.width", 140)
print(grp.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

ready = (grp["confidence"] == "High").sum()
med   = (grp["confidence"] == "Medium").sum()
short = (grp["confidence"] == "Insufficient").sum()
print(f"\nCategories: {ready} High, {med} Medium, {short} Insufficient")
if ready >= 5:
    print("  [OK] Enough High-confidence categories to make the table valuable.")
elif ready + med >= 6:
    print("  [~] Will show useful results if we let Medium-confidence cells render with a caveat.")
else:
    print("  [NO] Too few categories with sample size. Consider rolling up to project_size instead.")

## 4 — By assignee

Uses the task's `assignee` field as the "who". **Caveat:** a more accurate per-person multiplier would join tasks → time_entries and split `hours_worked` by who actually logged the time. We can do that in Step 1 if this audit shows the simpler version isn't accurate enough. Tasks with multiple assignees (comma-separated) are not split here.

In [ ]:
grp_p = usable.groupby("assignee").agg(
    n=("task_number", "count"),
    sum_est=("hours", "sum"),
    sum_actual=("hours_worked", "sum"),
    median_mult=("mult", "median"),
).reset_index()
grp_p["weighted_mult"] = grp_p["sum_actual"] / grp_p["sum_est"].replace(0, pd.NA)
grp_p["confidence"] = grp_p["n"].apply(confidence_tier)
grp_p = grp_p.sort_values("n", ascending=False).reset_index(drop=True)

print(grp_p.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

ready_p = (grp_p["confidence"] == "High").sum()
print(f"\nAssignees with High confidence: {ready_p} / {len(grp_p)}")

## 5 — Per-person × category cell coverage

The Phase 1 heatmap has roughly (active members) × (top ~10 categories) cells. We need to know what % of cells have n≥5 — if it's below ~30%, the heatmap will look mostly empty and we should swap it for a different view.

In [ ]:
cells = usable.groupby(["assignee", "category"]).size().rename("n").reset_index()
pivot = cells.pivot(index="assignee", columns="category", values="n").fillna(0).astype(int)

top_cats = grp.head(10)["category"].tolist()
pivot_top = pivot.reindex(columns=top_cats, fill_value=0)

print("Cell counts (assignee x top 10 categories):")
print(pivot_top.to_string())

n_cells = pivot_top.size
n_viable = (pivot_top >= N_MED).sum().sum()
n_high   = (pivot_top >= N_HIGH).sum().sum()
print(f"\nHeatmap cells: {n_cells} total ({pivot_top.shape[0]} ppl x {pivot_top.shape[1]} cats)")
print(f"  Viable (n>={N_MED}):       {n_viable:3d}  ({100*n_viable/max(n_cells,1):4.1f}%)")
print(f"  High conf (n>={N_HIGH}):    {n_high:3d}  ({100*n_high/max(n_cells,1):4.1f}%)")

if n_viable / max(n_cells, 1) >= 0.30:
    print("  [OK] Heatmap is worth building.")
else:
    print("  [~] Heatmap will be sparse. Consider per-person multiplier as a single column instead.")

## 6 — Outlier check

Multipliers above 10× or below 0.1× are almost always data quality problems — task taken way longer than estimated because the wrong number was entered, or someone logged time against the wrong task. We don't want these polluting the multipliers. Lists them so you can decide whether to fix the data or just exclude them in code.

In [ ]:
outliers = usable[(usable["mult"] > OUTLIER_HI) | (usable["mult"] < OUTLIER_LO)].copy()
outliers = outliers.sort_values("mult", ascending=False)
print(f"Outliers ({len(outliers)} tasks):")
cols = ["task_number", "project_number", "assignee", "category", "hours", "hours_worked", "mult", "title"]
if len(outliers):
    print(outliers[cols].to_string(index=False, float_format=lambda v: f"{v:.2f}"))
else:
    print("  None - clean data.")

share = len(outliers) / max(n_both, 1)
print(f"\nOutliers are {share*100:.1f}% of usable tasks.")
if share > 0.10:
    print("  [!] >10% - worth investigating data entry hygiene before building Phase 1.")
elif share > 0.03:
    print("  Modest - fine to exclude in code, but flag for cleanup.")
else:
    print("  Negligible - just exclude in Phase 1 code.")

## 7 — Project-level fallback

If task-level data is too thin, the same calibration math works at the project level: estimated duration (from `project_size` → `SIZE_DURATIONS`) vs actual duration (`actual_end - start_date`), and estimated effort (sum of task `hours`) vs actual effort (sum of task `hours_worked`). Projects are fewer but more complete. This section reports the project-level dataset shape so we can compare.

Uses the same defensive pull as the tasks layer.

In [ ]:
proj_layer = FeatureLayer(SOURCE_PROJECTS_URL, gis=gis)

print("Projects layer fields:")
for f in proj_layer.properties.fields:
    print(f"  {f['name']:28s} {f['type']}")
print()

pfset = proj_layer.query(where="1=1", out_fields="*", return_geometry=False)
projects = pd.DataFrame([f.attributes for f in pfset.features])
print(f"Pulled {len(projects)} total projects.")

proj_required = ["status", "actual_end", "start_date", "category", "project_size", "project_number"]
proj_missing  = [c for c in proj_required if c not in projects.columns]
if proj_missing:
    print(f"WARNING: missing expected project fields: {proj_missing}")
    print("Project layer columns:", list(projects.columns))
    raise KeyError(f"Update field name(s): {proj_missing}")

for c in ["start_date", "end_date", "actual_end"]:
    if c in projects.columns:
        projects[c + "_dt"] = pd.to_datetime(projects[c], unit="ms", errors="coerce")

projects_complete = projects[(projects["status"] == "Complete") & (projects["actual_end_dt"].notna()) & (projects["actual_end_dt"] >= CUTOFF)].reset_index(drop=True)
print(f"Completed projects in window: {len(projects_complete)}")

SIZE_DURATIONS = {"S": 2, "M": 6, "L": 13, "XL": 26}
projects_complete["planned_weeks"] = projects_complete["project_size"].map(SIZE_DURATIONS)
projects_complete["actual_weeks"]  = (projects_complete["actual_end_dt"] - projects_complete["start_date_dt"]).dt.days / 7

effort = recent.groupby("project_number").agg(
    proj_est_hours=("hours", "sum"),
    proj_actual_hours=("hours_worked", "sum"),
).reset_index()
projects_complete = projects_complete.merge(effort, on="project_number", how="left").fillna({"proj_est_hours": 0, "proj_actual_hours": 0})

usable_proj = projects_complete[(projects_complete["planned_weeks"].notna()) & (projects_complete["actual_weeks"] > 0)].copy()
usable_proj["duration_mult"] = usable_proj["actual_weeks"] / usable_proj["planned_weeks"]
print(f"\nUsable for duration calibration: {len(usable_proj)}")
if len(usable_proj):
    proj_grp = usable_proj.groupby("project_size").agg(
        n=("project_number", "count"),
        planned_wk_mean=("planned_weeks", "mean"),
        actual_wk_median=("actual_weeks", "median"),
        duration_mult_median=("duration_mult", "median"),
    ).reset_index()
    print(proj_grp.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

usable_proj_eff = usable_proj[(usable_proj["proj_est_hours"] > 0) & (usable_proj["proj_actual_hours"] > 0)].copy()
usable_proj_eff["effort_mult"] = usable_proj_eff["proj_actual_hours"] / usable_proj_eff["proj_est_hours"]
print(f"\nUsable for effort calibration: {len(usable_proj_eff)}")
if len(usable_proj_eff):
    print(f"  Project-level effort multiplier (hours-weighted): {usable_proj_eff['proj_actual_hours'].sum() / usable_proj_eff['proj_est_hours'].sum():.2f}x")

## 8 — Decision

After running everything above, the build decision falls into one of four buckets:

| Section 2 verdict | Section 3 verdict | Section 5 verdict | What to do |
|---|---|---|---|
| [OK] or [~] | ≥ 5 High cats | ≥ 30% viable cells | Build Phase 1 as mocked. |
| [OK] or [~] | ≥ 5 High cats | < 30% viable cells | Build Phase 1 *without* the per-person heatmap (or as a per-person single column). |
| [!] | Mixed | — | Drop the per-category table for now; show team-overall + project-level only. |
| [NO] | — | — | Phase 1 is premature. Step 1 becomes a data-entry hygiene push: make `hours` required on new tasks, backfill recent completions. |

Drop the actual outputs back into the conversation and we'll pick the bucket.